# PCA Feature Map Visualization

参考 GeoMetricLab 的可视化思路，这个 Notebook 用于：
1. 加载 SupScene 模型与可选权重
2. 提取 backbone patch tokens
3. 计算 PCA(3 通道) 特征图
4. 与原图叠加并保存结果

In [5]:
import os
import sys
import dataclasses
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from torchvision.transforms import v2 as T2

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Ensure relative paths in model/backbone loaders resolve from repo root.
os.chdir(PROJECT_ROOT)

from engine.conf import load_config
from supscene import create_encoder

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('cwd =', Path.cwd())
print('device =', device)

cwd = /home/sxl/project/SupScene
device = cuda


In [10]:
# ===== User settings =====
CONFIG_PATH = 'configs/peft-dinov2-scpp-lora.yaml'
WEIGHTS_PATH = 'weights/dinov2_scpp_supscene_1536.pth'
IMAGE_PATH = 'data/GL3D/test/5857aa5ab338a62ad5ff4dbe/images/00000004.jpg'
IMG_SIZE = 322
ALPHA = 0.6
SAVE_DIR = 'vis/pca_feature_map'

print(CONFIG_PATH, WEIGHTS_PATH, IMAGE_PATH)

configs/peft-dinov2-scpp-lora.yaml weights/dinov2_scpp_supscene_1536.pth data/GL3D/test/5857aa5ab338a62ad5ff4dbe/images/00000004.jpg


In [7]:
def build_model(config_path: str, weights_path: str | None):
    cfg = load_config(config_path, args=None)
    model_cfg = dataclasses.asdict(cfg.model)
    model_cfg['weights'] = None
    model = create_encoder(model_cfg)

    if weights_path:
        ckpt = torch.load(weights_path, map_location='cpu')
        state = ckpt.get('model_state_dict', ckpt.get('state_dict', ckpt))
        if not isinstance(state, dict):
            raise RuntimeError(f'Unsupported checkpoint format: {weights_path}')
        missing = model.load_state_dict(state, strict=False)
        print('Loaded checkpoint:', weights_path)
        print('missing keys:', len(getattr(missing, 'missing_keys', [])))
        print('unexpected keys:', len(getattr(missing, 'unexpected_keys', [])))

    model = model.to(device).eval()
    return model


def preprocess_image(image_path: str, img_size: int):
    image = Image.open(image_path).convert('RGB')
    tf = T2.Compose([
        T2.ToImage(),
        T2.Resize(size=(img_size, img_size), interpolation=T2.InterpolationMode.BICUBIC, antialias=True),
        T2.ToDtype(torch.float32, scale=True),
        T2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    x = tf(image).unsqueeze(0).to(device)
    return image, x


def normalize_patch_tokens(tokens: torch.Tensor) -> torch.Tensor:
    x = tokens.detach().float().cpu()
    if x.dim() == 4:
        if x.shape[0] == 1:
            x = x[0]
        c, h, w = x.shape
        x = x.reshape(c, h * w).transpose(0, 1)
    elif x.dim() == 3:
        if x.shape[0] == 1:
            x = x[0]
    if x.dim() != 2:
        raise ValueError(f'Unsupported patch token shape: {tuple(tokens.shape)}')

    n = x.shape[0]
    g = int(n ** 0.5)
    if g * g != n:
        raise ValueError(f'Patch count is not square: {n}')
    return x


def compute_pca_rgb(tokens: torch.Tensor) -> np.ndarray:
    x = normalize_patch_tokens(tokens)
    n = x.shape[0]
    g = int(n ** 0.5)

    x = x - x.mean(dim=0, keepdim=True)
    _, _, vh = torch.linalg.svd(x, full_matrices=False)
    z = x @ vh[:3].T
    z = z.reshape(g, g, 3)

    z_min = z.amin(dim=(0, 1), keepdim=True)
    z_max = z.amax(dim=(0, 1), keepdim=True)
    z = (z - z_min) / (z_max - z_min + 1e-6)
    return z.numpy()


def resize_pca(pca_rgb: np.ndarray, image: Image.Image) -> np.ndarray:
    p = torch.from_numpy(pca_rgb).permute(2, 0, 1).unsqueeze(0)
    r = F.interpolate(p, size=(image.size[1], image.size[0]), mode='bilinear', align_corners=False)[0]
    return r.permute(1, 2, 0).numpy().clip(0.0, 1.0)

In [ ]:
model = build_model(CONFIG_PATH, WEIGHTS_PATH)
raw_img, x = preprocess_image(IMAGE_PATH, IMG_SIZE)

with torch.no_grad():
    bb_out = model.backbone(x)

if isinstance(bb_out, tuple) and len(bb_out) >= 2:
    patch_tokens = bb_out[1]
elif torch.is_tensor(bb_out):
    patch_tokens = bb_out
else:
    raise RuntimeError(f'Backbone output does not contain patch tokens: type={type(bb_out)}')

pca_rgb = compute_pca_rgb(patch_tokens)
pca_up = resize_pca(pca_rgb, raw_img)
raw_np = np.asarray(raw_img).astype(np.float32) / 255.0
overlay = (1.0 - ALPHA) * raw_np + ALPHA * pca_up
overlay = np.clip(overlay, 0.0, 1.0)

fig, ax = plt.subplots(1, 3, figsize=(16, 6))
ax[0].imshow(raw_np); ax[0].set_title('Original'); ax[0].axis('off')
ax[1].imshow(pca_up); ax[1].set_title('PCA Feature Map'); ax[1].axis('off')
ax[2].imshow(overlay); ax[2].set_title('Overlay'); ax[2].axis('off')
plt.tight_layout(); plt.show()

✅ successfully loading config from: configs/peft-dinov2-scpp-lora.yaml


In [12]:
save_dir = Path(SAVE_DIR)
save_dir.mkdir(parents=True, exist_ok=True)
stem = Path(IMAGE_PATH).stem
pca_path = save_dir / f'{stem}_pca.png'
overlay_path = save_dir / f'{stem}_pca_overlay.png'

Image.fromarray((pca_up * 255).astype(np.uint8)).save(pca_path)
Image.fromarray((overlay * 255).astype(np.uint8)).save(overlay_path)

print('Saved:', pca_path)
print('Saved:', overlay_path)

NameError: name 'pca_up' is not defined